[Reference](https://pub.towardsai.net/ibms-granite-4-0-fine-tuning-made-simple-create-custom-ai-models-with-python-and-unsloth-4fc11b529c1f)

# Installing the packages

In [2]:
# Do this only in Colab notebooks! Otherwise use pip install unsloth
import re
import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")

In [3]:
!pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth

!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.5/283.5 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 15.2 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2025.11.4 requires msgspec, which is not installed.
unsloth-zoo 2025.11.4 requires tyro, which is not installed.
unsloth-zoo 2025.11.4 requires torchao>=0.13.0, but you have torchao 0.10.0 which is incompatible.
unsloth-zoo 2025.11.4 requires trl!=0.1

# Initialising Mamba Kernels

In [4]:
!pip install --no-build-isolation mamba_ssm==2.2.5
!pip install --no-build-isolation causal_conv1d==1.5.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.8/113.8 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 13.0 MB/s eta 0:00:00
  Created wheel for mamba_ssm: filename=mamba_ssm-2.2.5-cp312-cp312-linux_x86_64.whl size=532566033 sha256=c8b65fcabfb49a94456c9971619007218e4073f19a84fb6b3894f33d43bee4a1
  Stored in directory: /root/.cache/pip/wheels/21/55/c4/85b634055d6a9b599d27f5cbeacf353c6c532d8e2d8769960b
Successfully built mamba_ssm
  Preparing metadata (pyproject.toml) ... done
  Created wheel for causal_conv1d: filename=causal_conv1d-1.5.2-cp312-cp312-linux_x86_64.whl size=151160839 sha256=7cc4ae241543f93acfa4d96ecbc43e532f3aeed171f88c859f873f1a617e9606
  Stored in directory: /root/.cache/pip/wheels/b4/a5/a7/8d0ecdd7a890f633986e0e42bd99700dc256cbf33b67057f9f
Successfully built causal_conv1d


# Initialising model and tokenizer


In [5]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/granite-4.0-h-micro",
    max_seq_length = 1024,
    load_in_4bit = False,
    load_in_8bit = False,
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.3: Fast Granitemoehybrid patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.39G [00:00<?, ?B/s]

The fast path for GraniteMoeHybrid will be used when running the model on a GPU


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

# Adding LoRA Adapters

In [6]:
model = FastLanguageModel.get_peft_model(
    model, # The pretrained model
    r = 16, # The rank of the LoRA matrices
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",
                      "shared_mlp.input_linear", "shared_mlp.output_linear"],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407
)

Unsloth: Making `model.base_model.model.model` require gradients


# Data Preparation
## Loading the Dataset

In [7]:
from datasets import load_dataset, Dataset

sheet_url = "https://huggingface.co/datasets/unsloth/Support-Bot-Recommendation/raw/main/support_recs.csv"
dataset = load_dataset(
    "csv",
    data_files={"train": sheet_url},
    column_names=["snippet", "recommendation"],
    skiprows=1  # skip header rows
)["train"]

Generating train split: 0 examples [00:00, ? examples/s]

## Formatting the Dataset

In [8]:
def formatting_prompts_func(examples):
    user_texts = examples['snippet']
    response_texts = examples['recommendation']
    messages = [
        [{"role": "user", "content": user_text},
        {"role": "assistant", "content": response_text}] for user_text, response_text in zip(user_texts, response_texts)
    ]
    texts = [tokenizer.apply_chat_template(message, tokenize = False, add_generation_prompt = False) for message in messages]

    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/504 [00:00<?, ? examples/s]

# Training the Model

In [9]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model, # Our pretrained and initialised Model
    tokenizer = tokenizer, # Tokenizer of the model
    train_dataset = dataset, # Dataset to be used for finetuning
    eval_dataset = None, # Can be used to setup up evaluation
    args = SFTConfig(
        dataset_text_field = "text", # The field of the dataset that is structured, and will be used for training
        per_device_train_batch_size = 2, # Number of samples processed per device in each batch
        gradient_accumulation_steps = 4, # Number of steps to accumulate gradients before performing a backward pass
        warmup_steps = 5, # The number of steps for gradual learning rate increase at the start of training
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60, # The total number of training steps to perform
        learning_rate = 2e-4, # The learning rate for updating weights during training
        logging_steps = 1, # Frequency (in steps) for logging training metrics
        optim = "adamw_8bit", # Optimizer
        weight_decay = 0.01, # The regularization to prevent overfitting
        lr_scheduler_type = "linear", # To control learning rate decay
        seed = 3407, # Random state to help reproduce the same results everytime
        report_to = "none", # Platform for logging metrics can also be 'wandb'
    ),
)

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/504 [00:00<?, ? examples/s]

In [10]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_of_role|>user|end_of_role|>",
    response_part = "<|start_of_role|>assistant<|end_of_role|>",
)

Map (num_proc=6):   0%|          | 0/504 [00:00<?, ? examples/s]

In [11]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 504 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 19,202,048 of 3,210,598,144 (0.60% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.293000
2,1.572800
3,1.536400
4,2.051900
5,2.007600
6,2.151200
7,1.983500
8,2.046400
9,1.452900
10,1.343300


# Inferencing the Fine-Tuned Model

In [12]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# --- Scenario 1: Video-Conferencing Screen-Share Bug (11 turns) ---
scenario_1 = """
User: Everyone in my meeting just sees a black screen when I share.
Agent: Sorry about that—are you sharing a window or your entire screen?
User: Entire screen on macOS Sonoma.
Agent: Thanks. Do you have “Enable hardware acceleration” toggled on in Settings → Video?
User: Yeah, that switch is on.
Agent: Could you try toggling it off and start a quick test share?
User: Did that—still black for attendees.
Agent: Understood. Are you on the desktop app v5.4.2 or the browser client?
User: Desktop v5.4.2—just updated this morning.
"""

messages = [
    {"role": "user", "content": scenario_1},
]
inputs = tokenizer.apply_chat_template(
    messages, # List of messages created
    tokenize = True, # tokeniser
    add_generation_prompt = True, # Must add for generation
    padding = True, # to pad the inpur sequence to a uniform length
    return_tensors = "pt", # the PyTorch tensor as the output type
    return_dict=True, # Return the output as dictionary
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = False)

_ = model.generate(**inputs,
                   streamer = text_streamer,
                   max_new_tokens = 512, # Maximum number of tokens increase if tokens are getting cut off
                   use_cache = True, # Use of KV cache, which speeds up generation
                   # Adjust the sampling params to your preference
                   do_sample=True,
                   temperature = 0.7, top_p = 0.8, top_k = 20,
)

<|start_of_role|>system<|end_of_role|>You are a helpful assistant. Please ensure responses are professional, accurate, and safe.<|end_of_text|>
<|start_of_role|>user<|end_of_role|>
User: Everyone in my meeting just sees a black screen when I share.
Agent: Sorry about that—are you sharing a window or your entire screen?
User: Entire screen on macOS Sonoma.
Agent: Thanks. Do you have “Enable hardware acceleration” toggled on in Settings → Video?
User: Yeah, that switch is on.
Agent: Could you try toggling it off and start a quick test share?
User: Did that—still black for attendees.
Agent: Understood. Are you on the desktop app v5.4.2 or the browser client?
User: Desktop v5.4.2—just updated this morning.
<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>#### Analysis
The user is experiencing a black screen during screen sharing despite having hardware acceleration enabled and being on the latest desktop client version. This suggests a potential issue with the sharing mechanism or 

# Saving the Fine-Tuned Model

In [13]:
model.save_pretrained_merged("granite-4.0-h-micro-FINETUNED-16Bit", tokenizer, save_method = "merged_16bit")


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 2 files from cache to `granite-4.0-h-micro-FINETUNED-16Bit`: 100%|██████████| 2/2 [01:56<00:00, 58.14s/it]


Successfully copied all 2 files from cache to `granite-4.0-h-micro-FINETUNED-16Bit`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:24<00:00, 72.03s/it]


Unsloth: Merge process complete. Saved to `/content/granite-4.0-h-micro-FINETUNED-16Bit`


In [14]:
# model.push_to_hub_merged("Your_hf_ID/granite-4.0-h-micro-FINETUNED-16Bit", tokenizer, save_method = "merged_16bit", token = "YOUR_hf_TOKEN")